# 14 — Market and Operational Risk-Weighted Assets

## Completing the total RWA building blocks

Credit RWA is complete under both approaches. Before total RWA and the output floor are compared, the other two configured risk categories are shown separately.

Capital ratios use total RWA, not credit RWA alone. Market and operational RWA therefore enter both the standardised aggregate base and the model aggregate total.

### Non-credit risk terms used below

| Term | Simple meaning |
|---|---|
| Market RWA | RWA for trading-book and other market-risk positions |
| Operational RWA | RWA for losses from failed processes, people, systems or external events |
| BIC | Business Indicator Component in the operational-risk Standardised Approach |
| ILM | Internal Loss Multiplier based on operational-loss experience where applicable |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from basel_credit_risk.config import load_all_config
from basel_credit_risk.notebook_support import display, ensure_outputs

CRORE = 10_000_000
COLORS = ["#0B3A53", "#1F77B4", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"]
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
configs = load_all_config(ROOT / "config")
ensure_outputs(ROOT)
loans = pd.read_csv(ROOT / "data/processed/current_calculated.csv.gz", low_memory=False)
print(f"Portfolio represented: {len(loans):,} synthetic exposures")

Portfolio represented: 30,000 synthetic exposures


## Values used in this project

The two amounts below are synthetic configured inputs. They are not calculated from a synthetic trading book, financial statements or operational-loss database in this project.

In [2]:
non_credit = pd.DataFrame({
    "Risk category": ["Market risk", "Operational risk"],
    "RWA (INR crore)": [
        configs["capital_parameters"]["external_rwa"]["market_rwa"] / CRORE,
        configs["capital_parameters"]["external_rwa"]["operational_rwa"] / CRORE,
    ],
    "Source": ["Synthetic configured input", "Synthetic configured input"],
})
non_credit.loc[len(non_credit)] = ["Total non-credit RWA", non_credit["RWA (INR crore)"].sum(), "Sum"]
display(non_credit)

,Risk category,RWA (INR crore),Source
0,Market risk,11000.000000,Synthetic configured input
1,Operational risk,14000.000000,Synthetic configured input
2,Total non-credit RWA,25000.000000,Sum


INR 11,000 crore market RWA and INR 14,000 crore operational RWA produce INR 25,000 crore of non-credit RWA. These amounts remain unchanged across this project's credit stress scenarios.

## How a bank would calculate the two amounts

Market RWA is normally 12.5 times the applicable market-risk capital requirement. Under the applicable standardised framework, that requirement combines prescribed components such as sensitivities-based charges, default risk and residual risk. Operational RWA is 12.5 times operational-risk capital; under the Basel Standardised Approach, operational-risk capital equals BIC multiplied by ILM.

In [3]:
real_bank_route = pd.DataFrame([
    ["Market RWA", "Trading positions, sensitivities, risk factors, issuer defaults and residual risks", "12.5 × applicable market-risk capital requirement"],
    ["Operational RWA", "Three-year Business Indicator components and, where applicable, ten-year operational-loss history", "12.5 × BIC × ILM"],
], columns=["Output", "Data a bank needs", "Core route"])
display(real_bank_route)

,Output,Data a bank needs,Core route
0,Market RWA,"Trading positions, sensitivities, risk factors, issuer defaults and residual risks",12.5 × applicable market-risk capital requirement
1,Operational RWA,"Three-year Business Indicator components and, where applicable, ten-year operational-loss history",12.5 × BIC × ILM


The formulas explain how the configured figures would be replaced. The underlying trading, financial-statement and loss data are outside this project.

In [4]:
capital_charge_equivalent = non_credit.iloc[:2].copy()
capital_charge_equivalent["Equivalent capital charge (INR crore)"] = capital_charge_equivalent["RWA (INR crore)"] / 12.5
display(capital_charge_equivalent[["Risk category", "RWA (INR crore)", "Equivalent capital charge (INR crore)"]])

,Risk category,RWA (INR crore),Equivalent capital charge (INR crore)
0,Market risk,11000.000000,880.000000
1,Operational risk,14000.000000,1120.000000


The 12.5 conversion is the inverse of 8%. The displayed capital-charge equivalents explain the relationship only; they do not replace a full market-risk or operational-risk calculation.